In [2]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # put this at the very top of your script


from dataclasses import dataclass
from typing import Optional
import torch
import torch.nn.functional as F

from tools_llada import ConfKSorter, ConfCollectorInterface, BlockDiffusionQuotaHelper
from plugins_llada import InspectorPlugin




from transformers import AutoTokenizer
from torch.utils.data import DataLoader

from tools_llada import TopKSorter, TruthCollector
from plugins_llada import SaveKVPreviousPlugin_Disabled, CachePastKVPlugin_Disabled
from datasets import load_dataset, Dataset

from dataset_llada import LIST_DATASET
from datapreprocess_llada import parse_lines_with_index, merge_subdocs, PATTEN_REG_WIKI
from dataprocess_llada import Tokenizer_wiki_simple, Collater_wiki_simple

from modeling_yukai_llada import LLaDAModelLM

@dataclass
class DiffusionConfig:
    id_model: str
    len_prompt: int
    len_target: int    
    num_blocks: int
    num_unmask_per_step: int
    id_mask: int
    size_batch: int
    device: str
    klass_sorter: ConfKSorter
    klass_collector: ConfCollectorInterface
    klass_save_kv_previous: InspectorPlugin
    klass_cache_past_kv: InspectorPlugin
    
    size_block: Optional[int] = None
    step_per_block: Optional[int] = None

# end


config = DiffusionConfig(
    id_model='GSAI-ML/LLaDA-8B-Base',
    len_prompt=8,
    len_target=16,
    num_blocks=2,
    num_unmask_per_step=1,
    id_mask=126336,
    size_batch=1,
    device='cuda:0',
    klass_sorter=TopKSorter,
    klass_collector=TruthCollector,
    klass_save_kv_previous=SaveKVPreviousPlugin_Disabled,
    klass_cache_past_kv=CachePastKVPlugin_Disabled
)

config.size_block= int(config.len_target / config.num_blocks)
config.step_per_block=int(config.size_block / config.num_unmask_per_step)


'''load dataset first'''
name_dataset = LIST_DATASET[1]
ds = load_dataset(*name_dataset, split='all')
docs, _ = parse_lines_with_index(PATTEN_REG_WIKI, ds['text'])
docs = docs['subdocs']

samples = []
for doc in docs:
    lines_1 = doc['texts']
    paragraph_1 = ' '.join(lines_1)
    lines_remain, titles = merge_subdocs(doc['subdocs'])
    paragraph_remain = ' '.join(lines_remain)
    prefix = paragraph_1
    target = paragraph_remain
    samples.append({'text': paragraph_1 + ' ' + paragraph_remain})
# end

ds_origin = Dataset.from_list(samples[:3])


'''initialize constant hyper-parameters'''

'''load tokenizer'''
tokenizer = AutoTokenizer.from_pretrained(
    config.id_model,
    trust_remote_code=True
)

if tokenizer.padding_side != 'left':
    tokenizer.padding_side = 'left'
# end
assert tokenizer.pad_token_id != 126336




'''load model'''
model_kwargs = {}
model = LLaDAModelLM.from_pretrained(
    config.id_model,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    **model_kwargs
)

model = model.eval().to(config.device)


'''start to handle dataset based on hyper-parameter'''
len_max = config.len_prompt + config.len_target
ds = ds_origin\
    .filter(lambda x: x["text"] is not None and len(x["text"].strip()) > 0)\
    .map(Tokenizer_wiki_simple(tokenizer, len_max), remove_columns=ds_origin.column_names)\
    .filter(lambda x: x["length"] >= len_max)\
    .sort("length")
# end

'''prepare dataloader'''
loader = DataLoader(
    ds,
    batch_size=config.size_batch,
    shuffle=False,                 # keep sorted order
    collate_fn=Collater_wiki_simple(len_max, config.len_prompt, config.len_target, config.id_mask),
    drop_last=False
)

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
Filter: 100%|██████████| 3/3 [00:00<00:00, 3883.61 examples/s]


In [6]:
class SimpleLogitsSnapshot:

    def _regularize(self, sample, target):
        return  sample[:, :target.shape[1]]
    # end

    def __init__(self, logits, x, y, id_mask):
        self.id_mask = id_mask

        self.logits = logits

        self.x = self._regularize(x, logits)
        self.y = self._regularize(y, logits)

        self.x0 = torch.argmax(self.logits, dim=-1)

        self.p_finalized = torch.zeros(self.x.shape, dtype=torch.float64).to(self.x.device)
    # end

    def get_x(self):
        return self.x
    # end

    def get_p_finalized(self):
        return self.p_finalized
    # end

    def transform_logits(self, collector):

        logits_tranform = self.logits
        p = F.softmax(logits_tranform.to(torch.float64), dim=-1)

        index_p_all = collector.get_index(self)

        x0_p = torch.gather(p, dim=-1, index=index_p_all).squeeze(-1)

        neg_inf = torch.tensor(torch.finfo(x0_p.dtype).min, device=x0_p.device, dtype=x0_p.dtype)

        mask_mask = self.x == self.id_mask
        conf = torch.where(mask_mask, x0_p, neg_inf)  # (B, L)   # so only the masked part has confidence

        return conf
    # end

    def materialize_by_idx_(self, idx, conf):

        x0_target = torch.gather(self.x0, dim=-1, index=idx)
        conf_target = torch.gather(conf, dim=-1, index=idx)
        print('JINYUJ: materializing, conf_all: {}'.format(conf))
        print(f'JINYUJ: materializing idx: {idx} conf_target: {conf_target} x0:{x0_target}')

        self.x.scatter_(1, idx, x0_target)
        # print("JINYUJ in materialize_by_idx: scatter idx {} to x0 {}, self.x: {}".format(idx, x0_target, self.x))
        self.p_finalized.scatter_(1, idx, conf_target)
    # end


    def update_logits_(self, idx_logits, logits):
        logits_update = torch.gather(logits, dim=1, index=idx_logits)

        self.logits.scatter_(1, idx_logits, logits_update)
        x0 = torch.argmax(logits_update, dim=-1)
        self.x0.scatter_(1, idx_logits, x0)
    # end
# end


In [4]:

# 处理past_key_values
@ torch.no_grad()
def run_model_semi(model, x, y, config_diffusion):

    num_blocks = config_diffusion.num_blocks
    step_per_block = config_diffusion.step_per_block
    size_block = config_diffusion.size_block
    id_mask = config_diffusion.id_mask
    len_prompt = config_diffusion.len_prompt
    sorter = config_diffusion.klass_sorter()
    collector = config_diffusion.klass_collector()
    
    p_final = torch.zeros(x.shape, dtype=torch.float64, device=x.device)
    position_start = 0

    for id_block in range(1, num_blocks+1):
        position_end = position_start + len_prompt + id_block * size_block
        mask_mask_blk = x[:,position_start:position_end] == id_mask

        B = x.shape[0]
        L = position_end - position_start
        
        # idx_denoising = torch.arange(position_start, poition_end, dtype=torch.long).unsqueeze(0).expand(B, L).to(x.device)
        idx_denoising = torch.arange(position_start, position_end, dtype=torch.long).to(x.device)
        quota_helper = BlockDiffusionQuotaHelper(mask_mask_blk, size_block)

        for step in range(step_per_block):
            
            x_denoising,  y_denoising= x[:, idx_denoising], y[:, idx_denoising]
            logits = model(x_denoising, idx_current=idx_denoising).logits
            snapshot = SimpleLogitsSnapshot(logits, x_denoising, y_denoising, id_mask)
            conf_snapshot = snapshot.transform_logits(collector)
            idx_sorted_by_conf = sorter.argsort(conf_snapshot, snapshot)
            num_unmask = quota_helper.get_quota(step)

            idx_transform = idx_sorted_by_conf[:, :num_unmask]
            snapshot.materialize_by_idx_(idx_transform, conf_snapshot)
            
            x.scatter_(1, idx_transform, torch.gather(snapshot.get_x(), dim=1, index=idx_transform))
            p_final.scatter_(1, idx_transform, torch.gather(snapshot.get_p_finalized(), dim=1, index=idx_transform))
            
        # end for step
    # end for block

    return torch.gather(p_final, dim=-1, index=torch.arange(len_prompt, x.shape[-1], device=x.device, dtype=torch.long).unsqueeze(0).expand(x.shape[0], x.shape[1] - len_prompt))
# end

In [7]:
from tqdm import tqdm
from tools_llada import PPLCalculator

calculator = PPLCalculator()
model.fill_plugin(config.klass_cache_past_kv).fill_plugin(config.klass_save_kv_previous)

'''start the evaluation process'''
for id_batch, batch in enumerate(tqdm(loader)):
    conf = run_model_semi(
        model,
        batch['ids_prompt_masked_full'].to(config.device),
        batch['ids_target_masked_full'].to(config.device),
        config
    )

    print(calculator.cal(conf))
    break
# end for

  0%|          | 0/3 [00:00<?, ?it/s]

JINYUJ: materializing, conf_all: tensor([[-1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308,
         -1.7977e+308, -1.7977e+308, -1.7977e+308,   9.6299e-01,   8.8896e-01,
           6.4936e-02,   2.0354e-02,   3.0468e-03,   4.9336e-04,   3.9022e-04,
           1.1756e-04]], device='cuda:0', dtype=torch.float64)
JINYUJ: materializing idx: tensor([[8]], device='cuda:0') conf_target: tensor([[0.9630]], device='cuda:0', dtype=torch.float64) x0:tensor([[7510]], device='cuda:0')
JINYUJ: materializing, conf_all: tensor([[-1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308,
         -1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308,   9.9281e-01,
           1.9758e-02,   2.9828e-03,   8.9543e-04,   2.4445e-04,   7.1198e-04,
           1.1884e-04]], device='cuda:0', dtype=torch.float64)
JINYUJ: materializing idx: tensor([[9]], device='cuda:0') conf_target: tensor([[0.9928]], device='cuda:0', dtype=torch.float64) x0:tensor([[390]], device='cuda:0')
JINYU

  0%|          | 0/3 [00:00<?, ?it/s]

JINYUJ: materializing, conf_all: tensor([[-1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308,
         -1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308,
         -1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308,
         -1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308,
         -1.7977e+308,   2.3444e-07, -1.7977e+308,   1.3893e-06]],
       device='cuda:0', dtype=torch.float64)
JINYUJ: materializing idx: tensor([[23]], device='cuda:0') conf_target: tensor([[1.3893e-06]], device='cuda:0', dtype=torch.float64) x0:tensor([[198]], device='cuda:0')
JINYUJ: materializing, conf_all: tensor([[-1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308,
         -1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308,
         -1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308,
         -1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308, -1.7977e+308,
         -1.7977e+308